In [3]:
import torch
x = torch.tensor(2.0, requires_grad=True)
y = 3*x+2
y.backward()
print(x.grad)

tensor(3.)


In [4]:
x = torch.tensor(2.0, requires_grad=True)

a = 3 * x
y = a ** 2

y.backward()
print(x.grad)

tensor(36.)


In [5]:
x = torch.tensor(2.0, requires_grad=True)
y = torch.tensor(3.0, requires_grad=True)
z = x * y + 2
z.backward()
print(x.grad)
print(y.grad)

tensor(3.)
tensor(2.)


In [6]:
x = torch.tensor(2.0, requires_grad=True)

y = x ** 2
y.backward()

print(x.grad)

y = x ** 2
y.backward()

print(x.grad)

tensor(4.)
tensor(8.)


In [7]:
x = torch.tensor(2.0, requires_grad=True)

y = x * 3

print(x.is_leaf)
print(y.is_leaf)

True
False


In [ ]:
x = torch.tensor(2.0, requires_grad=True) 
a = 3 * x 
b = a ** 2 
b.retain_grad()    #  可以保存中间变量的梯度，一般只保存叶子节点的梯度
z = 2 * b + 1 
z.backward() 
print("b.grad =", b.grad)
print("z =", z) 
print("x.grad =", x.grad)

b.grad = tensor(2.)
z = tensor(73., grad_fn=<AddBackward0>)
x.grad = tensor(72.)


In [13]:
x = torch.tensor(2.0, requires_grad=True) 
a = 3 * x 
b = (x ** 2).detach()  # detach()可以将一个tensor从计算图中分离出来，分离出来的tensor不会被计算梯度

z = a+b
   
z.backward() 

print("x.grad =", x.grad)

x.grad = tensor(3.)


三种写法的对比
y = x.clone()

开辟新内存空间（修改 y 的数值不会影响 x）。

保留计算图（y 的梯度会继续回传给 x）。

y = x.detach()

共享内存空间（修改 y 的数值会直接改变 x）。

切断计算图（y 变成了叶子节点，梯度不会回传给 x）。

y = x.detach().clone()

开辟新内存空间（独立数据，互不影响）。

切断计算图（彻底切断联系，作为一个全新的独立 Tensor 使用）

In [ ]:
import torch

# -------------------------------------------------------------
# 场景 1：用 y = x.clone()
# 期望：修改 y 不影响 x，但求导时梯度能传回 x
# -------------------------------------------------------------
x1 = torch.tensor([2.0], requires_grad=True)
y1 = x1.clone()  # 深度复制数据，保留计算图

# 修改 y1 不会改变 x1
y1 = y1 * 3
z1 = y1 ** 2
z1.backward()

print("【clone】")
print("x1.grad:", x1.grad)  # 输出 tensor([12.]) -> 说明梯度能传回 x1

# -------------------------------------------------------------
# 场景 2：用 y = x.detach()
# 期望：切断梯度，但注意：y 和 x 共享内存！
# -------------------------------------------------------------
x2 = torch.tensor([2.0], requires_grad=True)
y2 = x2.detach()  # 切断梯度，共享内存

# 尝试原修改（In-place 操作）
y2.add_(1.0)  # 直接修改 y2 占用的内存

print("\n【detach】")
print("x2 的值被修改了:", x2)  # x2 也变成了 tensor([3.])！因为它们共享内存

# -------------------------------------------------------------
# 场景 3：用 y = x.detach().clone()
# 期望：既要数据独立，又要梯度切断（最安全的做法）
# -------------------------------------------------------------
x3 = torch.tensor([2.0], requires_grad=True)
y3 = x3.detach().clone()  # 完全独立的副本，且无计算图关系

y3.add_(1.0)  # 修改 y3
z3 = y3 ** 2

print("\n【detach().clone()】")
print("x3 原值不变:", x3)  # tensor([2.]) -> 修改 y3 不影响 x3
print("x3.grad:", x3.grad)  # None -> 梯度彻底切断，不会影响 x3

【clone】
x1.grad: tensor([36.])

【detach】
x2 的值被修改了: tensor([3.], requires_grad=True)

【detach().clone()】
x3 原值不变: tensor([2.], requires_grad=True)
x3.grad: None


In [ ]:
import torch
import matplotlib.pyplot as plt
# 生成数据
x = torch.linspace(-5, 5, 100).reshape(-1, 1)

true_w = 3.0
true_b = 2.0

y = true_w * x + true_b
losses = []
# 初始化参数
w = torch.randn(1, requires_grad=True)
b = torch.randn(1, requires_grad=True)

lr = 0.01
optimizer = torch.optim.SGD([w, b], lr=lr)

for epoch in range(1000):
    # 前向传播
    y_pred = w * x + b

    # 计算损失
    loss = ((y_pred - y) ** 2).mean()
    losses.append(loss.item())

    # 反向传播
    loss.backward()

    # 更新参数
    optimizer.step()
    optimizer.zero_grad()


    #print(f"Epoch {epoch+1}: loss={loss.item():.4f}, w={w.item():.4f}, b={b.item():.4f}")


Epoch 1: loss=134.2435, w=-0.2729, b=0.6000
Epoch 2: loss=93.0270, w=0.2836, b=0.6280
Epoch 3: loss=64.6133, w=0.7455, b=0.6555
Epoch 4: loss=45.0197, w=1.1288, b=0.6824
Epoch 5: loss=31.5024, w=1.4470, b=0.7087
Epoch 6: loss=22.1717, w=1.7111, b=0.7345
Epoch 7: loss=15.7257, w=1.9302, b=0.7599
Epoch 8: loss=11.2674, w=2.1121, b=0.7847
Epoch 9: loss=8.1791, w=2.2631, b=0.8090
Epoch 10: loss=6.0352, w=2.3884, b=0.8328
Epoch 11: loss=4.5426, w=2.4924, b=0.8561
Epoch 12: loss=3.4991, w=2.5787, b=0.8790
Epoch 13: loss=2.7656, w=2.6503, b=0.9014
Epoch 14: loss=2.2463, w=2.7098, b=0.9234
Epoch 15: loss=1.8751, w=2.7591, b=0.9449
Epoch 16: loss=1.6064, w=2.8001, b=0.9660
Epoch 17: loss=1.4089, w=2.8341, b=0.9867
Epoch 18: loss=1.2608, w=2.8623, b=1.0070
Epoch 19: loss=1.1473, w=2.8857, b=1.0268
Epoch 20: loss=1.0581, w=2.9051, b=1.0463
Epoch 21: loss=0.9860, w=2.9213, b=1.0654
Epoch 22: loss=0.9262, w=2.9347, b=1.0841
Epoch 23: loss=0.8752, w=2.9458, b=1.1024
Epoch 24: loss=0.8307, w=2.9550, 

In [ ]:
import torch
import matplotlib.pyplot as plt
# 生成数据
x = torch.linspace(-5, 5, 100).reshape(-1, 1)
print(x)


true_w = 3.0
true_b = 2.0

y = true_w * x + true_b   #这里也用到了pytorch的广播机制

y

tensor([[-5.0000],
        [-4.8990],
        [-4.7980],
        [-4.6970],
        [-4.5960],
        [-4.4949],
        [-4.3939],
        [-4.2929],
        [-4.1919],
        [-4.0909],
        [-3.9899],
        [-3.8889],
        [-3.7879],
        [-3.6869],
        [-3.5859],
        [-3.4848],
        [-3.3838],
        [-3.2828],
        [-3.1818],
        [-3.0808],
        [-2.9798],
        [-2.8788],
        [-2.7778],
        [-2.6768],
        [-2.5758],
        [-2.4747],
        [-2.3737],
        [-2.2727],
        [-2.1717],
        [-2.0707],
        [-1.9697],
        [-1.8687],
        [-1.7677],
        [-1.6667],
        [-1.5657],
        [-1.4646],
        [-1.3636],
        [-1.2626],
        [-1.1616],
        [-1.0606],
        [-0.9596],
        [-0.8586],
        [-0.7576],
        [-0.6566],
        [-0.5556],
        [-0.4545],
        [-0.3535],
        [-0.2525],
        [-0.1515],
        [-0.0505],
        [ 0.0505],
        [ 0.1515],
        [ 0.

tensor([[-13.0000],
        [-12.6970],
        [-12.3939],
        [-12.0909],
        [-11.7879],
        [-11.4848],
        [-11.1818],
        [-10.8788],
        [-10.5758],
        [-10.2727],
        [ -9.9697],
        [ -9.6667],
        [ -9.3636],
        [ -9.0606],
        [ -8.7576],
        [ -8.4545],
        [ -8.1515],
        [ -7.8485],
        [ -7.5455],
        [ -7.2424],
        [ -6.9394],
        [ -6.6364],
        [ -6.3333],
        [ -6.0303],
        [ -5.7273],
        [ -5.4242],
        [ -5.1212],
        [ -4.8182],
        [ -4.5152],
        [ -4.2121],
        [ -3.9091],
        [ -3.6061],
        [ -3.3030],
        [ -3.0000],
        [ -2.6970],
        [ -2.3939],
        [ -2.0909],
        [ -1.7879],
        [ -1.4848],
        [ -1.1818],
        [ -0.8788],
        [ -0.5758],
        [ -0.2727],
        [  0.0303],
        [  0.3333],
        [  0.6364],
        [  0.9394],
        [  1.2424],
        [  1.5455],
        [  1.8485],


In [ ]:
import torch

# =====================
# 实验 1：梯度累积
# =====================

x = torch.tensor(2.0, requires_grad=True)

y = 3 * x
z = y ** 2

z.backward(retain_graph=True)

print("第一次 backward")
print("x.grad =", x.grad)  # 36

z.backward()

print("第二次 backward，不清零")
print("x.grad =", x.grad)  # 72


# =====================
# 实验 2：清零梯度
# =====================

x = torch.tensor(2.0, requires_grad=True)

y = 3 * x
z = y ** 2

z.backward(retain_graph=True)

print("第一次")
print("x.grad =", x.grad)  # 36

x.grad.zero_()

z.backward()

print("清零之后第二次")
print("x.grad =", x.grad)  # 36


# =====================
# 实验 3：非叶子梯度
# =====================

x = torch.tensor(2.0, requires_grad=True)

y = 3 * x
z = y ** 2

print("x.is_leaf =", x.is_leaf)  # True
print("y.is_leaf =", y.is_leaf)  # False

y.retain_grad()

z.backward()

print("x.grad =", x.grad)  # 36
print("y.grad =", y.grad)  # 12

# 对，y 是中间变量，但“中间变量”和 requires_grad=True 完全不冲突。 这里特别容易把两个概念混在一起。

假设：

x = torch.ones(1, 4)          # requires_grad=False
fc = nn.Linear(4, 4)

y = fc(x)

Linear 本质上计算：

$$ y=xW^T+b $$

虽然：

x.requires_grad == False

但是 Linear 的参数默认：

fc.weight.requires_grad == True
fc.bias.requires_grad == True

于是计算关系是：

x (requires_grad=False)
        │
        │
W (requires_grad=True)
        │
b (requires_grad=True)
        │
        ▼
        y
        │
requires_grad=True
为什么中间变量 y 也要是 True？

因为 PyTorch 需要通过 y 继续往回计算 W 和 b 的梯度。

例如后面还有：

y = fc(x)
loss = y.sum()

loss.backward()

反向传播需要走：

                 forward
x ──────┐
        │
W ──────┼──→ y ───→ loss
        │
b ──────┘


                 backward
W.grad ←──── y ←──── loss
b.grad ←────┘

为了能够从 loss 一路反向传播到 W、b，PyTorch 必须知道 y 是计算图的一部分。

所以：

print(y.requires_grad)
# True
一个非常重要的区别

你可能把：

“中间变量默认不会保存 .grad”

和：

“中间变量 requires_grad=False”

混在一起了。

这两个完全不同。

例如：

x = torch.tensor(2.0, requires_grad=True)

y = x * 3
z = y ** 2

z.backward()

print(x.requires_grad)   # True
print(y.requires_grad)   # True

print(x.grad)            # tensor(36.)
print(y.grad)            # 通常是 None

这里很关键：

              requires_grad       .grad 默认保存？

x 叶子节点          True                 ✅
      │
      ▼
y 中间节点          True                 ❌
      │
      ▼
z

y.requires_grad=True 不代表 y.grad 一定会被保存。

requires_grad=True 更接近于：

这个 Tensor 所处的计算需要被 Autograd 跟踪，因为反向传播可能需要经过它。

而 .grad 是：

这个 Tensor 的梯度最终有没有被保存在 .grad 属性中。

默认情况下，PyTorch 主要把 .grad 累积到需要梯度的**叶子 Tensor（leaf tensor）**上，比如模型参数。

你可以亲手做这个实验
import torch

x = torch.tensor(2.0, requires_grad=True)

y = x * 3
z = y ** 2

print("x.requires_grad:", x.requires_grad)
print("y.requires_grad:", y.requires_grad)
print("z.requires_grad:", z.requires_grad)

print("x.is_leaf:", x.is_leaf)
print("y.is_leaf:", y.is_leaf)

z.backward()

print("x.grad:", x.grad)
print("y.grad:", y.grad)

你会看到核心现象：

x.requires_grad: True
y.requires_grad: True
z.requires_grad: True

x.is_leaf: True
y.is_leaf: False

x.grad: tensor(36.)
y.grad: None

所以你现在可以把这三个概念分开：

属性	意思
requires_grad=True	Autograd 需要追踪与该 Tensor 相关的可求导计算
is_leaf=True	它是不是计算图中的叶子 Tensor
.grad	反向传播后是否在这里保存/累积梯度
再回到刚才的 y = model(x)

即使：

x.requires_grad = False

但只要模型里面有：

W.requires_grad = True
b.requires_grad = True

那么：

x ───────────┐
             │
W [True] ────┼──→ y [requires_grad=True]
             │
b [True] ────┘

y 是中间结果没错，但它来自需要梯度的参数，所以它属于计算图，y.requires_grad=True。

而当你写：

with torch.no_grad():
    y = model(x)

相当于告诉 PyTorch：

这段计算不要建立反向传播计算图

于是：

print(y.requires_grad)
# False

这也正是为什么前面的实验能够证明 model.eval() 和 torch.no_grad() 是两回事。

#  grad_fn   里面的AddmmBackward0 是什么意思？

这个更有意思。

你的网络里有：

self.fc = nn.Linear(4, 4)

Linear 本质上是在计算：

$$ y=xW^T+b $$

也就是：

矩阵乘法 + 加 bias

PyTorch 内部可以把这种操作实现为类似 addmm 的运算：

add + matrix multiplication

所以 y.grad_fn 显示：

<AddmmBackward0 ...>

可以简单理解成：

PyTorch 记得 y 是通过一个“矩阵乘法 + 加法”相关的可求导操作得到的，并保存了对应的反向传播节点。

3. grad_fn 到底是什么？

这是你现在最值得理解的地方。

比如：

x = torch.tensor(2.0, requires_grad=True)

y = x * 3
z = y ** 2

计算图：

x
│
│ × 3
▼
y
│
│ ** 2
▼
z

PyTorch 不仅保存 Tensor，还要知道：

y 是怎么来的？
z 是怎么来的？

所以可能：

print(y.grad_fn)

得到类似：

<MulBackward0>

因为：

y = x * 3

是乘法产生的。

而：

print(z.grad_fn)

可能得到：

<PowBackward0>

因为：

z = y ** 2

是幂运算产生的。

所以你可以暂时把：

grad_fn

理解成：

这个 Tensor 在计算图中是由什么可求导操作产生的，以及反向传播要从这里接着怎么往前走。

4. 为什么你的结果是 AddmmBackward0？

因为你的：

y = model(x)

最终经过了：

nn.Linear

Linear 大致：

$$ y=xW^T+b $$

因此计算图大致是：

              W
              │
              ▼
x ───────→ 矩阵乘法
              │
b ───────→   +
              │
              ▼
              y

于是：

y.grad_fn

告诉你：

AddmmBackward0

也就是：

这个 y 是由 Addmm 相关运算产生的
如果以后 backward()
我知道应该怎么从这里继续向前计算梯度
5. 后面的 0x0000024BBFBEB490 是什么？

这个不用管。

<AddmmBackward0 object at 0x0000024BBFBEB490>

其中：

AddmmBackward0

才是你关心的。

后面的：

0x0000024BBFBEB490

只是这个 Python 对象在当前运行过程中的标识/地址表示，每次运行都可能不同，没有学习意义，不需要记。

6. 现在做最关键的对照实验

你刚才已经得到：

model.eval()

y = model(x)

print(y.requires_grad)
print(y.grad_fn)

结果：

True
<AddmmBackward0 ...>

现在只改一个地方：

model.eval()

with torch.no_grad():
    y = model(x)

print(y.requires_grad)
print(y.grad_fn)

你应该会得到：

False
None

把两次放一起看：

代码	requires_grad	grad_fn
model.eval()	True	AddmmBackward0
model.eval() + no_grad()	False	None

这就是一个非常漂亮的实验结果。

第一种：

model.eval()

Linear
  ↓
y
  ↓
requires_grad = True
grad_fn = AddmmBackward0

说明：

模型虽然进入 eval 模式，但 Autograd 仍然在追踪计算。

第二种：

model.eval()

with torch.no_grad():

    Linear
      ↓
      y

得到：

requires_grad = False
grad_fn = None

说明：

no_grad() 才让这段前向传播不进行梯度追踪。

你现在可以把三个概念彻底分开
requires_grad
    ↓
这次计算是否涉及需要梯度追踪的结果

grad_fn
    ↓
这个非叶子 Tensor 是由什么可求导操作产生的

.grad
    ↓
backward 后实际存放/累积下来的梯度

所以千万不要看到：

y.requires_grad = True

就理解成：

“y 里面已经有梯度了。”

不是。

更准确地说是：

“与 y 有关的这条计算路径正在被 Autograd 追踪，因此之后可以通过它进行反向传播。”

只有执行：

loss.backward()

之后，我们才真正进入“根据计算图计算梯度”的阶段。

你现在对 requires_grad → grad_fn → backward → .grad 这条链理解清楚之后，训练循环里的 loss.backward() 就会一下子变得非常直观。